# ACDSA 2027 Paper — Validation and Reporting Audit

**Purpose.** This notebook is a post-processing and validation layer for the completed
`reproduction_run` experiment. It does **not** modify the optimization, caches,
DESIGN freeze, selected layouts, or TEST evaluations.

It systematically:

1. verifies that reported layouts come from the frozen DESIGN set;
2. audits rare recovered cases with non-finite PEB;
3. preserves the original strict optimization rule;
4. computes a transparent reporting metric:
   - finite-case median PEB95;
   - number and percentage of non-finite recovered PEB cases;
5. checks seed and structured-search outputs;
6. exports publication-ready audit tables.

### Scientific rule

The completed optimization used the conservative rule:

> if any recovered case had non-finite PEB, aggregate PEB quality was reported as `NaN`
> and treated as `+inf` for optimization ordering.

This notebook does **not** retroactively change that rule.

For final descriptive reporting only, it additionally computes the median PEB over the
finite recovered cases and reports the non-finite fraction separately. This keeps numerical
failures visible without allowing a tiny number of pathological cases to erase the median
of the valid recovered population.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

import os
RUN_ROOT = Path(os.environ.get("HAPS_GNSS_RUN_OUTPUT", "outputs/reproduction_run")).resolve()
RAW = RUN_ROOT / "raw"
CACHE = RUN_ROOT / "cache"
OUT = RUN_ROOT / "publication_audit"
OUT.mkdir(parents=True, exist_ok=True)

assert RUN_ROOT.exists(), f"Run root not found: {RUN_ROOT}"
assert RAW.exists(), f"Raw folder not found: {RAW}"

print("Run root:", RUN_ROOT)
print("Audit output:", OUT)

## 1. Load experiment outputs

These files are read exactly as produced by the completed run. Nothing in `raw/` is overwritten.

In [ ]:
FILES = {
    "headline": RAW / "camera_ready_headline_results.csv",
    "frozen": RAW / "frozen_design_layouts.csv",
    "design_reps": RAW / "paper_representatives_design_selected.csv",
    "test_selected": RAW / "test_results_design_selected.csv",
    "seed_reps": RAW / "seed_representatives_design_selected.csv",
    "seed_test": RAW / "seed_test_statistics.csv",
    "structured_fronts": RAW / "structured_design_fronts.csv",
    "structured_decisions": RAW / "structured_refinement_decisions.csv",
    "reference": RAW / "fixed_reference_results.csv",
    "common_pop": RAW / "common_population_peb_ecdf.csv",
    "design_population": RAW / "design_population_summary.csv",
    "urban": RAW / "urban_geometry_summary.csv",
}

missing = [str(p) for p in FILES.values() if not p.exists()]
assert not missing, "Missing expected files:\n" + "\n".join(missing)

data = {name: pd.read_csv(path) for name, path in FILES.items()}

for name, df in data.items():
    print(f"{name:22s} {df.shape}")

## 2. DESIGN-freeze integrity

TEST must never choose layouts. Every final representative and every TEST-evaluated layout
must already belong to the frozen DESIGN set.

In [ ]:
frozen = data["frozen"].copy()
reps = data["design_reps"].copy()
test_selected = data["test_selected"].copy()

frozen_ids = set(frozen["placement_id"].astype(str))
missing_rep_ids = sorted(set(reps["placement_id"].astype(str)) - frozen_ids)
missing_test_ids = sorted(set(test_selected["placement_id"].astype(str)) - frozen_ids)

print("DESIGN representatives missing from frozen set:", missing_rep_ids)
print("TEST-evaluated layouts missing from frozen set:", missing_test_ids)

assert not missing_rep_ids
assert not missing_test_ids

print("[OK] All final/tested layouts belong to the frozen DESIGN set.")

## 3. Frozen layouts with undefined DESIGN PEB

The original strict rule can yield `design_f2 = NaN` when even one recovered case has
non-finite PEB. This cell identifies whether such layouts survived into the frozen set.

In [ ]:
affected_frozen = frozen[frozen["design_f2"].isna()].copy()

print("Frozen DESIGN layouts:", len(frozen))
print("Frozen layouts with design_f2 = NaN:", len(affected_frozen))

cols = [c for c in [
    "roi_id","algorithm","seed","placement_id","design_f1","design_f2"
] if c in affected_frozen.columns]

affected_frozen[cols].sort_values(["roi_id","algorithm","placement_id"])

## 4. Full DESIGN-cache audit

For each city, quantify how often non-finite recovered PEB occurred across all cached DESIGN evaluations.
This does not change any original score.

In [ ]:
CITY_IDS = ["shinjuku_tokyo", "downtown_dubai", "frankfurt_bankenviertel"]

cache_all = {}
rows_out = []

for rid in CITY_IDS:
    path = CACHE / rid / "placement_eval" / "design.jsonl"
    assert path.exists(), f"Missing DESIGN cache: {path}"

    rows = [json.loads(line) for line in path.open() if line.strip()]
    cache_all[rid] = rows

    bad = [r for r in rows if int(r.get("n_nonfinite_peb",0) or 0) > 0]
    good = [r for r in rows if int(r.get("n_nonfinite_peb",0) or 0) == 0]

    rows_out.append({
        "roi_id": rid,
        "n_design_evaluations": len(rows),
        "n_with_nonfinite_peb": len(bad),
        "pct_with_nonfinite_peb": 100*len(bad)/len(rows) if rows else np.nan,
        "max_nonfinite_peb_count": max([int(r.get("n_nonfinite_peb",0) or 0) for r in bad], default=0),
        "max_R_finite_quality": max([float(r["f1"]) for r in good], default=np.nan),
        "max_R_nonfinite_quality": max([float(r["f1"]) for r in bad], default=np.nan),
    })

design_cache_audit = pd.DataFrame(rows_out)
design_cache_audit

## 5. Exact DESIGN audit of affected frozen layouts

Cache records may not contain `placement_id`, so affected frozen layouts are matched by their
four placement angles and radii.

In [ ]:
def arrays_from_frozen(row):
    th = row[[f"theta_{i}" for i in range(1,5)]].to_numpy(float)
    rr = row[[f"r_{i}_m" for i in range(1,5)]].to_numpy(float)
    return th, rr

def match_cache_record(row):
    rid = row["roi_id"]
    th, rr = arrays_from_frozen(row)
    for rec in cache_all[rid]:
        rth = np.asarray(rec.get("thetas",[]), float)
        rrr = np.asarray(rec.get("rs",[]), float)
        if len(rth)==4 and len(rrr)==4:
            if np.allclose(rth, th, rtol=1e-10, atol=1e-8) and np.allclose(rrr, rr, rtol=1e-10, atol=1e-8):
                return rec
    return None

audit_rows = []
for _, row in affected_frozen.iterrows():
    rec = match_cache_record(row)
    assert rec is not None, f"Could not match {row['placement_id']}"

    n_rec = int(rec.get("n_p2",0) or 0)
    n_bad = int(rec.get("n_nonfinite_peb",0) or 0)

    audit_rows.append({
        "roi_id": row["roi_id"],
        "algorithm": row["algorithm"],
        "seed": row.get("seed",""),
        "placement_id": row["placement_id"],
        "design_R": float(rec["f1"]),
        "design_recovered": n_rec,
        "design_nonfinite_PEB": n_bad,
        "design_nonfinite_PEB_pct": 100*n_bad/n_rec if n_rec else np.nan,
        "original_design_PEB": rec.get("f2",np.nan),
        "original_design_f2_order": rec.get("f2_order",np.nan),
    })

affected_design_audit = pd.DataFrame(audit_rows)
affected_design_audit

## 6. Receiver-level TEST PEB audit

For every TEST detail file, compute:
- recovered cases;
- finite/non-finite PEB counts;
- non-finite percentage;
- finite-case median/P95/max PEB95.

These are reporting diagnostics only.

In [ ]:
def summarize_detail(path):
    d = pd.read_csv(path)
    assert {"made_solvable","peb95_m"}.issubset(d.columns), path

    rec = d.loc[d["made_solvable"].astype(bool)].copy()
    peb = pd.to_numeric(rec["peb95_m"], errors="coerce").to_numpy(float)
    finite_mask = np.isfinite(peb)
    finite = peb[finite_mask]

    n_rec = len(peb)
    n_fin = int(finite_mask.sum())
    n_bad = n_rec - n_fin

    return {
        "n_recovered": n_rec,
        "n_finite_PEB": n_fin,
        "n_nonfinite_PEB": n_bad,
        "nonfinite_PEB_pct": 100*n_bad/n_rec if n_rec else np.nan,
        "finite_case_median_PEB95_m": float(np.median(finite)) if n_fin else np.nan,
        "finite_case_P95_PEB95_m": float(np.percentile(finite,95)) if n_fin else np.nan,
        "finite_case_max_PEB95_m": float(np.max(finite)) if n_fin else np.nan,
    }

detail_rows = []
for rid in CITY_IDS:
    folder = RAW / rid / "test_checkpoint"
    assert folder.exists(), folder

    for path in sorted(folder.glob("*_detail.csv")):
        s = summarize_detail(path)
        s.update({
            "roi_id": rid,
            "placement_id": path.name.replace("_detail.csv",""),
            "detail_file": str(path),
        })
        detail_rows.append(s)

detail_summary = pd.DataFrame(detail_rows)

# Reference detail files and headline results use different placement identifiers.
# Match the reference rows through a dedicated join key for consistent reporting.
detail_summary["headline_placement_id"] = detail_summary["placement_id"].replace(
    {"preserved_c4_reference": "c4_300m_24h_phase"}
)

print("TEST detail files audited:", len(detail_summary))
detail_summary.sort_values(["roi_id","placement_id"]).head(20)

## 7. Publication reporting table

The original headline table is preserved. This derived table adds finite-case PEB reporting and
the non-finite diagnostic.

Where the original strict median is finite, both medians must agree exactly (within floating-point tolerance).

In [ ]:
headline = data["headline"].copy()

detail_for_join = detail_summary[[
    "roi_id","headline_placement_id",
    "n_recovered","n_finite_PEB","n_nonfinite_PEB",
    "nonfinite_PEB_pct",
    "finite_case_median_PEB95_m",
    "finite_case_P95_PEB95_m",
    "finite_case_max_PEB95_m",
]].copy()

report = headline.merge(
    detail_for_join,
    left_on=["roi_id","placement_id"],
    right_on=["roi_id","headline_placement_id"],
    how="left",
    validate="many_to_one"
).drop(columns=["headline_placement_id"])

mask = (
    report["test_conditional_median_PEB95_m"].notna()
    & report["finite_case_median_PEB95_m"].notna()
)

if mask.any():
    delta = np.abs(
        report.loc[mask,"test_conditional_median_PEB95_m"].to_numpy(float)
        - report.loc[mask,"finite_case_median_PEB95_m"].to_numpy(float)
    )
    print("Maximum median difference where original median is finite:", np.nanmax(delta))
    assert np.nanmax(delta) < 1e-8

show = [c for c in [
    "roi_id","algorithm","seed","placement_id","selection_rule",
    "design_f1","design_f2","test_R",
    "test_conditional_median_PEB95_m",
    "finite_case_median_PEB95_m",
    "n_recovered","n_nonfinite_PEB","nonfinite_PEB_pct"
] if c in report.columns]

report[show].sort_values(["roi_id","algorithm","selection_rule"])

## 8. Cases where strict TEST PEB is undefined

These are the rows for which final descriptive reporting should use:

**finite-case median PEB95 + explicit non-finite percentage**

instead of leaving an unexplained `NaN`.

In [ ]:
strict_nan = report[
    report["test_conditional_median_PEB95_m"].isna()
    & report["n_recovered"].fillna(0).gt(0)
].copy()

cols = [c for c in [
    "roi_id","algorithm","placement_id","selection_rule",
    "test_R","n_recovered","n_finite_PEB","n_nonfinite_PEB",
    "nonfinite_PEB_pct","finite_case_median_PEB95_m"
] if c in strict_nan.columns]

strict_nan[cols].sort_values(["roi_id","algorithm"])

## 9. Representative-selection structure

Expected interpretation:
- Greedy: recovery-optimized representative only.
- DE: recovery-optimized representative only.
- NSGA-II: DESIGN max-\(R\) and DESIGN min-PEB representatives.
- Symmetric ring: DESIGN max-\(R\) and DESIGN min-PEB representatives.
- Center+3: DESIGN max-\(R\) and DESIGN min-PEB representatives.
- Fixed reference: deterministic benchmark layout.

In [ ]:
selection_structure = (
    headline.groupby(["algorithm","selection_rule"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["algorithm","selection_rule"])
)
selection_structure

## 10. Seed variability

Use the already generated seed-level statistics as stochastic robustness evidence for DE and NSGA-II.
TEST is not used to choose seeds retrospectively.

In [ ]:
seed_stats = data["seed_test"].copy()
print(seed_stats.shape)
seed_stats

## 11. Common-population quality output

This metric is the fairest cross-layout quality comparison because all layouts are evaluated
against the same GNSS-unsolved denominator.

In [ ]:
common_pop = data["common_pop"].copy()
print("shape:", common_pop.shape)
print("columns:", common_pop.columns.tolist())
common_pop.head(20)

## 12. Structured refinement decisions

This verifies the realized structured-search path. In the completed run, all six city/family
combinations entered Stage A and Stage B.

In [ ]:
structured_decisions = data["structured_decisions"].copy()
structured_decisions

## 13. Export audit package

These are derived publication files. Raw experiment outputs remain untouched.

In [ ]:
report_path = OUT / "camera_ready_reporting_table_with_peb_audit.csv"
affected_path = OUT / "affected_frozen_design_layouts.csv"
cache_path = OUT / "design_cache_nonfinite_audit.csv"
detail_path = OUT / "test_detail_peb_audit.csv"
strict_nan_path = OUT / "strict_nan_reporting_cases.csv"

report.to_csv(report_path, index=False)
affected_design_audit.to_csv(affected_path, index=False)
design_cache_audit.to_csv(cache_path, index=False)
detail_summary.to_csv(detail_path, index=False)
strict_nan.to_csv(strict_nan_path, index=False)

summary = {
    "run_root": str(RUN_ROOT),
    "n_frozen_design_layouts": int(len(frozen)),
    "n_frozen_layouts_with_design_f2_nan": int(len(affected_frozen)),
    "n_test_detail_files_audited": int(len(detail_summary)),
    "n_headline_rows_with_strict_test_peb_nan_and_recovery": int(len(strict_nan)),
    "reporting_rule": (
        "Preserve original strict optimization score. For descriptive reporting only, "
        "report finite-case median PEB95 together with non-finite recovered count/percentage."
    ),
    "raw_outputs_modified": False,
}
summary_path = OUT / "AUDIT_SUMMARY.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("[OK] Exported:")
for p in [report_path, affected_path, cache_path, detail_path, strict_nan_path, summary_path]:
    print(" -", p)

## 14. Interpretation and reporting

If all assertions pass:

- keep the original optimization rule exactly as executed;
- report TEST recovery \(R\) for the frozen DESIGN-selected layouts;
- for quality, report finite-case median PEB95 and the non-finite recovered fraction when non-zero;
- emphasize the common-population quality curve for fair cross-layout comparison;
- do not select layouts, seeds, thresholds, or methods using TEST performance.